# J-Space Prompt-Injection Layer Selection — Colab

This is the thin Colab launcher for **Phase 1 only**. Reusable logic lives in the `jspace_research.phase1` package. The pipeline builds frozen BIPIA train/validation pairs, captures final-prompt-token residuals, reconstructs sparse J-space, learns one clean-to-injection direction per fitted layer, and selects the layer with the highest validation macro-AUPRC.

It does not implement detector thresholds, test or transfer evaluation, behavioral generation, interventions, gating, utility evaluation, or hidden-state baselines.

## 1. Install the experiment and pinned BIPIA checkout

In [ ]:
import subprocess
from pathlib import Path

RESEARCH_REPO = 'https://github.com/ethanncyb/jspace-research.git'
RESEARCH_REVISION = 'prompt-injection-experiment'
JLENS_REVISION = '581d398613e5602a5af361e1c34d3a92ea82ba8e'
BIPIA_REVISION = 'a004b69ec0dd446e0afd461d98cb5e96e120a5d0'
REPO_ROOT = Path('/content/jspace-research')
BIPIA_CHECKOUT = Path('/content/BIPIA')

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', RESEARCH_REVISION, RESEARCH_REPO, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', RESEARCH_REVISION], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', RESEARCH_REVISION], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'merge', '--ff-only', f'origin/{RESEARCH_REVISION}'], check=True)
if not BIPIA_CHECKOUT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/microsoft/BIPIA.git', str(BIPIA_CHECKOUT)], check=True)
subprocess.run(['git', '-C', str(BIPIA_CHECKOUT), 'checkout', BIPIA_REVISION], check=True)
subprocess.run(['pip', 'install', '-q', '-e', str(REPO_ROOT)], check=True)
print('Research revision:', subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip())
print('BIPIA revision:', subprocess.check_output(['git', '-C', str(BIPIA_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())
print('Jacobian-lens pin:', JLENS_REVISION)

## 2. Authenticate and configure paths

In [ ]:
from huggingface_hub import notebook_login

# Gemma or the released lens may require an accepted license and authenticated session.
notebook_login()

In [ ]:
RUN_MODE = 'smoke'  # change to 'full' only after smoke succeeds
OUTPUT_DIR = Path('/content/jspace_phase1_smoke' if RUN_MODE == 'smoke' else '/content/jspace_phase1_full')
BIPIA_ROOT = BIPIA_CHECKOUT / 'benchmark'

# Required only for RUN_MODE='full'. These files must already be in BIPIA train.jsonl format.
WEBQA_TRAIN_PATH = None
SUMMARIZATION_TRAIN_PATH = None

CONFIG_PATH = REPO_ROOT / 'configs' / f'phase1_{RUN_MODE}.yaml'
print('Config:', CONFIG_PATH)
print('Output:', OUTPUT_DIR)

## 3. Run Phase 1

`all` is resumable. To separate the expensive model load from decomposition, run the same command with `--stage prepare`, then `capture`, then `analyze`.

In [ ]:
command = [
    'jspace-phase1',
    '--config', str(CONFIG_PATH),
    '--bipia-root', str(BIPIA_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--stage', 'all',
]
if WEBQA_TRAIN_PATH is not None:
    command.extend(['--webqa-train', str(WEBQA_TRAIN_PATH)])
if SUMMARIZATION_TRAIN_PATH is not None:
    command.extend(['--summarization-train', str(SUMMARIZATION_TRAIN_PATH)])

process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'Phase 1 failed with exit status {return_code}; see the traceback above.')

## 4. Inspect the frozen Phase 1 result

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

print(json.dumps(json.loads((OUTPUT_DIR / 'selected_layer.json').read_text()), indent=2))
display(pd.read_csv(OUTPUT_DIR / 'layer_metrics.csv'))
display(Image(filename=str(OUTPUT_DIR / 'layer_auprc.png')))
display(Image(filename=str(OUTPUT_DIR / 'selected_layer_score_distribution.png')))

## Interpretation boundary

The selected layer is the fitted layer whose shared sparse-J-space mean-difference direction best separates held-out BIPIA attack/control prompts by macro-AUPRC. This result alone does not establish causality, behavioral recognition, transfer, defense effectiveness, or superiority over a full-residual detector.